# Training a Potential: how to setup data pipeline, model and trainer 

# Training a Nerual Network Potential

This section introduces how to use `MolPot` to train a nnp

## Preparing and loding data

Before we can start training neural networks with `MolPot`, we need to prepare our data. 

In [1]:
%load_ext autoreload
%autoreload 2

import molpot as mpot
import torch
from ignite.metrics import MeanAbsoluteError, BatchWise
from pathlib import Path

/opt/conda/lib/python3.12/site-packages/ignite/handlers/checkpoint.py:16: DeprecationWarning: `TorchScript` support for functional optimizers is deprecated and will be removed in a future PyTorch release. Consider using the `torch.compile` optimizer instead.
  from torch.distributed.optim import ZeroRedundancyOptimizer


In [11]:
# 1. get rMD17 dataset
rmd17_ds = mpot.dataset.rMD17(
    molecule="aspirin",
    save_dir="data",
    device="cpu"
)
rmd17_ds.add_process(mpot.process.NeighborList(cutoff=5.0))

In [14]:
data_inspector = mpot.inspector.DataInspector(rmd17_ds)
data_inspector.summary()

number of data: 1000

                     dataset: rMD17                     
┏━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┓
┃ label  ┃      type       ┃    unit    ┃   comment    ┃
┡━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━┩
│ energy │ <class 'float'> │  kcal/mol  │ total energy │
│ forces │ <class 'float'> │ kcal/mol/A │  all forces  │
└────────┴─────────────────┴────────────┴──────────────┘

In [15]:
train_ds, valid_ds = torch.utils.data.random_split(rmd17_ds, [.95, .05])

train_dl = mpot.DataLoader(
    dataset=train_ds,
    batch_size=10,
    shuffle=False,
)
eval_dl = mpot.DataLoader(
    dataset=valid_ds,
    batch_size=10,
    shuffle=False,
)

## Setting up the model

In [16]:
pinet = mpot.potential.nnp.PiNet(
    depth=5,
    basis_fn=mpot.potential.nnp.radial.GaussianRBF(10, 5.0),
    cutoff_fn=mpot.potential.nnp.cutoff.CosineCutoff(5.0),
    pi_nodes=[64, 64],
    ii_nodes=[64, 64, 64, 64],
    pp_nodes=[64, 64, 64, 64],
    activation=torch.nn.Tanh(),
    rank=1
)
e_readout = mpot.potential.nnp.readout.Atomwise(
    [64, 1],
    from_key=("pinet", "p1"),
    to_key=("predicts", "energy")
)
f_readout = mpot.potential.nnp.readout.DPairPot(
    fx_key=("predicts", "energy"),
    dx_key=("pairs", "dist"),
    to_key=("predicts", "forces"),
    create_graph=True
)
model = mpot.potential.PotentialSeq("pinet", pinet, e_readout, f_readout)

In [17]:
save_dir = Path("pinet2-rmd17")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.99)
loss_fn = mpot.engine.MultiTargetLoss(
    torch.nn.MSELoss(), [("energy", "energy", 1.0), ("forces", "forces", 10.0)]
)

potential_trainer = mpot.PotentialTrainer(
    model = model,
    optimizer = scheduler,
    loss_fn = loss_fn,
    device = "cpu",
    no_grad_eval = True,
)
potential_trainer.add_lr_scheduler()
potential_trainer.add_checkpoint(save_dir)
potential_trainer.add_metric("e_mae", MeanAbsoluteError(lambda pred, label: (pred["energy"], label["energy"])), BatchWise(), target="all")
potential_trainer.add_metric("f_mae", MeanAbsoluteError(lambda pred, label: (pred["forces"], label["forces"])), BatchWise(), target="all")
potential_trainer.attach_tensorboard(
    save_dir/"tb",
)
state = potential_trainer.run(train_data=train_dl, max_steps=1000, eval_data=eval_dl)

Current run is terminating due to exception: Could not run 'neighbors::getNeighborPairs' with arguments from the 'CPU' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'neighbors::getNeighborPairs' is only available for these backends: [Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMeta, Tracer, AutocastCPU, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGradWrapper, PythonTLSSnapshot, FuncTorchDynamicLayerFrontMode, PreDispatch, PythonDispatcher].

Meta: registered at ../aten/src/AT

NotImplementedError: Could not run 'neighbors::getNeighborPairs' with arguments from the 'CPU' backend. This could be because the operator doesn't exist for this backend, or was omitted during the selective/custom build process (if using custom build). If you are a Facebook employee using PyTorch on mobile, please visit https://fburl.com/ptmfixes for possible resolutions. 'neighbors::getNeighborPairs' is only available for these backends: [Meta, BackendSelect, Python, FuncTorchDynamicLayerBackMode, Functionalize, Named, Conjugate, Negative, ZeroTensor, ADInplaceOrView, AutogradOther, AutogradCPU, AutogradCUDA, AutogradXLA, AutogradMPS, AutogradXPU, AutogradHPU, AutogradLazy, AutogradMeta, Tracer, AutocastCPU, AutocastXPU, AutocastMPS, AutocastCUDA, FuncTorchBatched, BatchedNestedTensor, FuncTorchVmapMode, Batched, VmapMode, FuncTorchGradWrapper, PythonTLSSnapshot, FuncTorchDynamicLayerFrontMode, PreDispatch, PythonDispatcher].

Meta: registered at ../aten/src/ATen/core/MetaFallbackKern